In [11]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Importa a classe da sua rede neural desenvolvida do zero
from Neural_Network import NeuralNetwork

# ==========================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
# ==========================================
print("Carregando e preparando os dados...")
df = pd.read_csv(r'C:\Users\guilh\Downloads\Mobile_Price_Classification-main\Mobile_Price_Classification-main\Database\train.csv')

# Extrair matriz de covariáveis X e vetor alvo y
X = df.drop(columns=["price_range"]).values.astype(float)
y = df["price_range"].values

# Normalização Min-Max (Fundamental para a Rede Neural)
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_norm = (X - X_min) / (X_max - X_min + 1e-8)

# Divisão Holdout Clássica: 70% Treino / 30% Validação
split = int(0.7 * len(X_norm))
X_tr, X_val = X_norm[:split], X_norm[split:]
y_tr, y_val = y[:split], y[split:]

print(f"Tamanho do Treino: {X_tr.shape[0]} observações")
print(f"Tamanho da Validação: {X_val.shape[0]} observações\n")

# ==========================================
# 2. TREINANDO O MODELO 1: REDE NEURAL (MLP)
# ==========================================
print("--- Iniciando o treino da Rede Neural ---")
# 20 variáveis de entrada -> camadas ocultas -> 4 saídas (classes)
nn = NeuralNetwork(layer_sizes=[20, 16, 8, 4], seed=127)

# Treinamento usando gradiente descendente
nn.train(X_tr, y_tr, epochs=10000, learning_rate=0.01, verbose=False)

# Para prever na validação, fazemos o forward pass (Propagação Direta)
ativacoes_val, _ = nn.forward(X_val)
probabilidades_nn = ativacoes_val[-1] # Saída da Softmax, estima \eta(x)

# Seleciona a classe com maior probabilidade estimada
predicoes_nn = np.argmax(probabilidades_nn, axis=1)


# ==========================================
# 3. TREINANDO O MODELO 2: RANDOM FOREST
# ==========================================
print("--- Iniciando o treino do Random Forest ---")
# Usando as árvores descorrelacionadas (como visto na disciplina)
# Parâmetros otimizados anteriormente: 300 árvores e max_features=0.5
rf = RandomForestClassifier(n_estimators=300, max_features=0.5, random_state=42)
rf.fit(X_tr, y_tr)

# Previsão sobre a mesma base de validação
predicoes_rf = rf.predict(X_val)


# ==========================================
# 4. AVALIAÇÃO DO RISCO EMPÍRICO (COMPARAÇÃO)
# ==========================================
print("\n" + "="*50)
print("             RESULTADOS FINAIS")
print("="*50)

acc_nn = accuracy_score(y_val, predicoes_nn)
acc_rf = accuracy_score(y_val, predicoes_rf)

print(f"Acurácia Global - Rede Neural  : {acc_nn * 100:.2f}%")
print(f"Acurácia Global - Random Forest: {acc_rf * 100:.2f}%\n")

print("--- Relatório Detalhado: REDE NEURAL ---")
print(classification_report(y_val, predicoes_nn))

print("--- Relatório Detalhado: RANDOM FOREST ---")
print(classification_report(y_val, predicoes_rf))

Carregando e preparando os dados...
Tamanho do Treino: 1400 observações
Tamanho da Validação: 600 observações

--- Iniciando o treino da Rede Neural ---
--- Iniciando o treino do Random Forest ---

             RESULTADOS FINAIS
Acurácia Global - Rede Neural  : 57.17%
Acurácia Global - Random Forest: 89.00%

--- Relatório Detalhado: REDE NEURAL ---
              precision    recall  f1-score   support

           0       0.68      0.87      0.76       154
           1       0.52      0.17      0.25       161
           2       0.41      0.35      0.38       147
           3       0.58      0.94      0.72       138

    accuracy                           0.57       600
   macro avg       0.55      0.58      0.53       600
weighted avg       0.55      0.57      0.52       600

--- Relatório Detalhado: RANDOM FOREST ---
              precision    recall  f1-score   support

           0       0.95      0.94      0.94       154
           1       0.85      0.89      0.87       161
        